# lineup on Kaggle

**Settings (right panel):** Accelerator -> **GPU T4 x2**, Internet -> **On**. Then **Save Version -> Save & Run All (Commit)** for a run that survives a disconnect.

The run **prints its config and self-checks loudly**: every step prints `[ok] ...`, prints steady progress (`... methods 75/300`) so a long phase never looks frozen, and stops with `!!!!!! STOP -- ... !!!!!!` if anything is wrong. Watch the early log for `hard_traps=` and `decoy chunks=` to confirm you're running what you intend. Work is saved after each stage; the results zip is self-labelled by config (e.g. `lineup_2wiki_hardtraps_qwen14.zip`) in the Output panel.

In [ ]:
import os, sys
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # 14B 4-bit needs the unfragmented allocator to fit one T4
if not os.path.isdir("/kaggle/working/LINEUP"):
    !git clone -q https://github.com/santoshcheethiralame-dot/LINEUP /kaggle/working/LINEUP
%cd /kaggle/working/LINEUP
!pip install -q bitsandbytes
!pip install -q -e . --no-deps
if "/kaggle/working/LINEUP/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/LINEUP/src")
import lineup, torch
print("lineup", lineup.__version__, "| cuda:", torch.cuda.is_available())

## Settings

The defaults are tuned for solid results (limit 400, 32 ablations, full abstention) and fit one commit for the two small models. Change a few things per experiment:

- **Harder traps:** set `HARD_TRAPS = True` (adds a redundant decoy per case).
- **Cross-dataset:** set `DATASET = "2wiki"`. If the split errors in the first minute, try `SPLIT = "dev"` or `SPLIT = "test"`.
- **Bigger model:** set `MODELS = ["qwen14"]` and `N_ABLATIONS = 24`, and give it its own commit (14B is ~2x slower, so one model per commit stays under the 12h limit).

In [ ]:
LIMIT = 400
K = 6
SPLIT = "validation"
DATASET = "hotpotqa"     # or "2wiki"
SEED = 0
MAX_NEW_TOKENS = 32
N_ABLATIONS = 32         # stable ContextCite; drop to 24 for the 14B model to fit one commit
METHODS_SCOPE = "all"    # "all" scores the abstention margins too; "wrong" is ~2x faster
HARD_TRAPS = False       # True adds a redundant decoy per case (the "harder traps" experiment)
N_DECOYS = 1 if HARD_TRAPS else 0   # decoys per case: 0 baseline, 1 hard-traps, 2-3 = dose-response sweep
NATURAL = False   # True = no planting; gold + the dataset's own distractors (set N_DECOYS=0)

MODEL_ZOO = {
    "qwen":   "Qwen/Qwen2.5-7B-Instruct",
    "phi":    "microsoft/Phi-3.5-mini-instruct",
    "qwen14": "Qwen/Qwen2.5-14B-Instruct",
    "mistral": "mistralai/Mistral-7B-Instruct-v0.3",
}
MODELS = ["qwen", "phi"]   # keys from MODEL_ZOO to run this commit; for the big model use ["qwen14"] alone

In [ ]:
from pathlib import Path

import torch

from lineup.backends import TransformersModel
from lineup.config import set_seed
from lineup.correctness import LLMJudge
from lineup.data.scenario import ScenarioBuilder
from lineup.data.schema import CaseRoles
from lineup.data.serialization import write_generations, write_predictions, write_roles, write_scenarios
from lineup.data.sources import load_examples
from lineup.data.substitution import build_answer_pool
from lineup.generation import generate_and_judge
from lineup.methods import ContextCite, LexicalSimilarity, LLMJudgeCulprit, SingleChunkSupport, run_method
from lineup.oracle import leave_one_out


def check(ok, message):
    if not ok:
        raise AssertionError("\n!!!!!! STOP -- " + message + " !!!!!!\n")
    print("  [ok] " + message)


def progress(name, phase, i, total, every=25):
    if (i + 1) % every == 0 or (i + 1) == total:
        print(f"  {name}: {phase} {i + 1}/{total}", flush=True)


print("=" * 72)
print(f" CONFIG  dataset={DATASET}  limit={LIMIT}  k={K}  seed={SEED}")
print(f"         n_ablations={N_ABLATIONS}  methods_scope={METHODS_SCOPE}  hard_traps={HARD_TRAPS}  n_decoys={N_DECOYS}  natural={NATURAL}")
print(f"         models={MODELS}")
print("=" * 72)
check(torch.cuda.is_available(), "GPU is available" if torch.cuda.is_available() else "NO GPU -- set Accelerator to GPU T4 x2")
check(DATASET in ("hotpotqa", "2wiki"), f"DATASET valid ({DATASET})")
check(METHODS_SCOPE in ("wrong", "all"), f"METHODS_SCOPE valid ({METHODS_SCOPE})")
check(all(name in MODEL_ZOO for name in MODELS), f"MODELS valid ({MODELS})")

set_seed(SEED)
examples = list(load_examples(DATASET, SPLIT, limit=LIMIT))
check(len(examples) >= 50, f"loaded enough examples from {DATASET} ({len(examples)})")
builder = ScenarioBuilder(answer_pool=build_answer_pool(examples), k=K, seed=SEED, n_decoys=N_DECOYS, natural=NATURAL)
scenarios = [s for s in (builder.build(e) for e in examples) if s is not None]
decoys = sum(1 for s in scenarios for c in s.chunks if c.provenance == "decoy")
print(f"\n built {len(scenarios)} cases from {len(examples)} questions  |  decoy chunks={decoys}")
check(len(scenarios) >= 50, f"enough cases built ({len(scenarios)})")
check(decoys == N_DECOYS * len(scenarios), f"decoys match N_DECOYS ({decoys} == {N_DECOYS} x {len(scenarios)})")

for name in MODELS:
    model_id = MODEL_ZOO[name]
    print(f"\n{'-' * 72}\n MODEL: {name}  ({model_id})\n{'-' * 72}")
    out = Path("runs") / name
    out.mkdir(parents=True, exist_ok=True)
    write_scenarios(out / "scenarios.jsonl", scenarios)
    set_seed(SEED)
    model = TransformersModel(model_id, max_new_tokens=MAX_NEW_TOKENS, load_in_4bit=True)
    judge = LLMJudge(model)

    generations = []
    for i, s in enumerate(scenarios):
        generations.append(generate_and_judge(model, s, llm_judge=judge))
        progress(name, "generate", i, len(scenarios))
    write_generations(out / "generations.jsonl", generations)
    n_wrong = sum(not g.is_correct for g in generations)
    print(f" {name}: {n_wrong} wrong of {len(scenarios)}")
    check(n_wrong >= 10, f"{name} has enough wrong cases to score ({n_wrong})")

    role_cases = []
    pairs = list(zip(scenarios, generations))
    for i, (s, g) in enumerate(pairs):
        if g.is_correct:
            role_cases.append(CaseRoles(s.qid, s.question, s.gold_answer, g.model_answer, True, []))
        else:
            role_cases.append(leave_one_out(model, s, g, llm_judge=judge))
        progress(name, "oracle", i, len(pairs))
    write_roles(out / "roles.jsonl", role_cases)

    methods = [ContextCite(n_ablations=N_ABLATIONS, seed=SEED), LexicalSimilarity(), LLMJudgeCulprit(), SingleChunkSupport()]
    method_cases = [(s, g) for s, g in zip(scenarios, generations) if METHODS_SCOPE == "all" or not g.is_correct]
    predictions = []
    for i, (s, g) in enumerate(method_cases):
        for m in methods:
            predictions.append(run_method(m, model, s, g.model_answer))
        progress(name, "methods", i, len(method_cases))
    write_predictions(out / "predictions.jsonl", predictions)

    for fn in ("scenarios", "generations", "roles", "predictions"):
        p = out / f"{fn}.jsonl"
        check(p.exists() and p.stat().st_size > 0, f"{name}/{fn}.jsonl written ({p.stat().st_size if p.exists() else 0} bytes)")
    print(f" [DONE] {name}", flush=True)

    del model, judge
    torch.cuda.empty_cache()

print(f"\n{'=' * 72}\n ALL MODELS COMPLETE\n{'=' * 72}")

## Results

The summary below prints the headline numbers straight to the log, then the four-panel dashboard: misleading-as-culprit with 95% intervals, the predicted-role heatmap, the risk-coverage curves, and the cross-model agreement.

In [ ]:
from pathlib import Path

from lineup.data.serialization import read_predictions, read_roles
from lineup.scoring import score_predictions
from lineup.setvalued import attribution_recovery


def fmt(value):
    return f"{value:.2f}" if value is not None else "n/a"


for name in MODELS:
    roles = read_roles(Path("runs") / name / "roles.jsonl")
    wrong = [c for c in roles if not c.original_correct]
    preds = read_predictions(Path("runs") / name / "predictions.jsonl")
    print(f"\n== {name}  (n_wrong={len(wrong)}, dataset={DATASET}, n_decoys={N_DECOYS}) ==")
    print(" attribution accuracy:")
    for r in score_predictions(wrong, preds):
        print(f"   {r.method:18s} n_culprit={r.n_with_culprit:3d}  top1={fmt(r.top1_culprit_accuracy)}  "
              f"misleading-as-culprit={r.misleading_as_culprit_rate:.2f}  culprit>misleading={fmt(r.culprit_over_misleading_winrate)}")
    print(" set recovery (top-1 vs top-|R|) and self-reliability:")
    for r in attribution_recovery(wrong, preds):
        print(f"   {r.method:18s} |R|={r.mean_responsible:.2f}  recall@1={fmt(r.recall_at_1)}  "
              f"recall@k={fmt(r.recall_at_k)}  reliability_auroc={fmt(r.reliability_auroc)}  single_culprit_auroc={fmt(r.single_culprit_auroc)}")

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

from lineup.agreement import compare_models
from lineup.data.serialization import read_generations, read_predictions, read_roles
from lineup.downstream import abstention_curves
from lineup.scoring import bootstrap_intervals, score_predictions

ROLE_ORDER = ["culprit", "misleading", "silent", "inert"]


def render_dashboard(run_dirs, seed=0, n_boot=2000):
    data = {}
    for name, directory in run_dirs.items():
        base = Path(directory)
        roles = read_roles(base / "roles.jsonl")
        wrong = [case for case in roles if not case.original_correct]
        preds = read_predictions(base / "predictions.jsonl")
        data[name] = {
            "roles": roles, "preds": preds, "gens": read_generations(base / "generations.jsonl"),
            "reports": {r.method: r for r in score_predictions(wrong, preds)},
            "cis": bootstrap_intervals(wrong, preds, n_boot=n_boot, seed=seed),
        }

    methods = sorted(next(iter(data.values()))["reports"])
    names = list(run_dirs)
    primary = names[0]
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    ax = axes[0, 0]
    width = 0.8 / len(names)
    for i, name in enumerate(names):
        reps, cis = data[name]["reports"], data[name]["cis"]
        vals = [reps[m].misleading_as_culprit_rate for m in methods]
        lo, hi = [], []
        for j, m in enumerate(methods):
            low, high = cis[m]["misleading_as_culprit_rate"]
            lo.append(vals[j] - (low if low is not None else vals[j]))
            hi.append((high if high is not None else vals[j]) - vals[j])
        ax.bar([x + i * width for x in range(len(methods))], vals, width, yerr=[lo, hi], capsize=4, label=name)
    ax.set_xticks([x + width * (len(names) - 1) / 2 for x in range(len(methods))])
    ax.set_xticklabels(methods, rotation=20, ha="right")
    ax.set_ylabel("misleading-as-culprit rate")
    ax.set_title("How often each method blames the near-miss")
    ax.set_ylim(0, 1)
    ax.legend()

    ax = axes[0, 1]
    reps = data[primary]["reports"]
    matrix = [[reps[m].predicted_role_rate.get(r, 0.0) for r in ROLE_ORDER] for m in methods]
    image = ax.imshow(matrix, cmap="magma", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(ROLE_ORDER)))
    ax.set_xticklabels(ROLE_ORDER)
    ax.set_yticks(range(len(methods)))
    ax.set_yticklabels(methods)
    for r in range(len(methods)):
        for c in range(len(ROLE_ORDER)):
            ax.text(c, r, f"{matrix[r][c]:.2f}", ha="center", va="center",
                    color="white" if matrix[r][c] < 0.5 else "black", fontsize=9)
    ax.set_title(f"Where {primary}'s predicted culprit truly lands")
    fig.colorbar(image, ax=ax, fraction=0.046)

    ax = axes[1, 0]
    for signal, (cov, risk) in abstention_curves(data[primary]["gens"], data[primary]["preds"], data[primary]["roles"]).items():
        if cov:
            ax.plot(cov, risk, label=signal, linewidth=1.5)
    ax.set_xlabel("coverage")
    ax.set_ylabel("risk (error of the answered set)")
    ax.set_title("Selective answering -- does attribution help abstain")
    ax.legend(fontsize=8)

    ax = axes[1, 1]
    ax.axis("off")

    def fmt(value):
        return f"{value:.2f}" if value is not None else "n/a"

    if len(names) >= 2:
        rep = compare_models(data[names[0]]["roles"], data[names[1]]["roles"])
        lines = [
            f"cross-model agreement: {names[0]} vs {names[1]}", "",
            f"cases wrong in both:     {rep.n_both_wrong} / {rep.n_common}",
            f"same culprit set:        {fmt(rep.same_culprit_rate)}",
            f"culprit-set Jaccard:     {fmt(rep.culprit_jaccard)}",
            f"per-passage role kappa:  {fmt(rep.role_kappa)}",
        ]
        ax.text(0.0, 0.95, "\n".join(lines), va="top", family="monospace", fontsize=12)
    else:
        ax.text(0.0, 0.95, "add a second model to fill this panel", va="top", fontsize=12)

    fig.tight_layout()
    fig.savefig("runs/dashboard.png", dpi=130, bbox_inches="tight")
    plt.show()


render_dashboard({name: f"runs/{name}" for name in MODELS}, seed=SEED, n_boot=2000)

In [ ]:
import shutil
from pathlib import Path

ok = True
for name in MODELS:
    for fn in ("scenarios", "generations", "roles", "predictions"):
        p = Path("runs") / name / f"{fn}.jsonl"
        if not (p.exists() and p.stat().st_size > 0):
            print(f" MISSING: {p}")
            ok = False
print("\n" + "=" * 72)
print(" ALL OUTPUTS PRESENT -- safe to download" if ok else " SOMETHING IS MISSING -- DO NOT TRUST THESE RESULTS")
print("=" * 72)
cond = "natural" if NATURAL else "baseline" if N_DECOYS == 0 else "hardtraps" if N_DECOYS == 1 else f"dose{N_DECOYS}"
tag = DATASET + "_" + cond + "_" + "-".join(MODELS)
shutil.make_archive(f"/kaggle/working/lineup_{tag}", "zip", "runs")
print(f"download lineup_{tag}.zip from the Output panel on the right")
assert ok, "run incomplete -- a file is missing"